# 1 · See the same EEG three ways
 **Goal:** turn real EEG into a reproducible visual object, then discover what the renderer hides.
 You will use subject S001: R01 eyes open and R02 eyes closed. No classifier is trained here.
 **Predict:** which view would make a sustained oscillation easiest to notice: waveform or time–frequency map?

## Before running
Use Python 3.11 or 3.12 in Colab or local Jupyter. Code is provided: **predict → run → inspect → explain**.
Downloads and training happen only when **you** run the relevant cells. This source notebook has no precomputed results.
Use the [PhysioNet release](https://physionet.org/content/eegmmidb/1.0.0/) under its Open Data Commons Attribution License v1.0.
Cite Schalk (2009), DOI 10.13026/C28G6P, Schalk et al. (2004), DOI 10.1109/TBME.2004.827072,
and the platform citation listed on that release. Data remain outside Git.
The helper is downloaded from this course repository; inspect/review its source before running.
The default moving `main` makes starting easy; pin `SOURCE_REF` to a reviewed commit for a durable submission.
Save the exact helper bytes/hash, source notebook, package versions, configuration, and outputs.

In [ ]:
%pip -q install "numpy>=1.26,<3" "scipy>=1.11,<2" "mne>=1.7,<2" "matplotlib>=3.8,<4" "pandas>=2,<3" "Pillow>=10,<13" "scikit-learn>=1.4,<2" "ipywidgets>=8,<9"

In [ ]:
from pathlib import Path
import hashlib, importlib.util, json, urllib.request
SOURCE_REF = "main"  # For a repeatable handoff, replace with a reviewed student-repository commit SHA.
helper = Path("chapter18_vision_studio.py")
url = ("https://raw.githubusercontent.com/ZagElSayed/Introduction-to-Neuroinformatics/"
       + SOURCE_REF + "/chapters/ch18-vision-ai/vision_studio.py")
payload = urllib.request.urlopen(url, timeout=60).read()
helper.write_bytes(payload)  # Keep this exact helper with your evidence; do not rely on a moving branch.
HELPER_HASH = hashlib.sha256(payload).hexdigest()
spec = importlib.util.spec_from_file_location("chapter18_vision_studio", helper)
studio = importlib.util.module_from_spec(spec)
spec.loader.exec_module(studio)
print("Helper SHA-256:", HELPER_HASH)
print(studio.versions())

In [ ]:
from datetime import datetime, timezone
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = Path("outputs/ch18_preview") / RUN_TAG
data = studio.prepare_public_data([1], OUTPUT_DIR)
studio.write_json(OUTPUT_DIR / "helper_source.json", {"url": url, "ref": SOURCE_REF, "sha256": HELPER_HASH})
(OUTPUT_DIR / "vision_studio_source.py").write_bytes(payload)
print(data["manifest"][["derivative_id", "condition", "window_start_s"]].head())
print("Power map:", data["power"].shape, "Pixels:", data["pixels"].shape)

## Inspect the representation
Each four-second window has 256 samples per channel at 64 Hz. A 64-sample symmetric Hann window and
16-sample hop give 13 time columns and 33 one-sided frequency bins; retaining 2–30 Hz leaves 29 rows.
Average the three channels' **power matrices**, preserving posterior power without claiming a scalp map.
The model pixels use fixed −20 to 40 dB bounds relative to 1 microvolt²/Hz, with high frequencies at the top.
The display has labels and color; the saved model input is an undecorated grayscale image.

In [ ]:
fig = studio.show_views(data, 0)
fig.savefig(OUTPUT_DIR / "eyes_open_views.png", dpi=150, bbox_inches="tight")
fig = studio.show_views(data, 10)
fig.savefig(OUTPUT_DIR / "eyes_closed_views.png", dpi=150, bbox_inches="tight")

In [ ]:
import ipywidgets as widgets
from IPython.display import display
def explore(index=0, low=-20, high=40, palette="viridis"):
    if low >= high:
        print("Choose lower bound < upper bound."); return
    studio.show_views(data, index, (low, high), palette)
display(widgets.interactive(explore,
    index=widgets.IntSlider(min=0,max=19,value=0,description="Window"),
    low=widgets.IntSlider(min=-40,max=10,value=-20,description="dB low"),
    high=widgets.IntSlider(min=20,max=60,value=40,description="dB high"),
    palette=widgets.Dropdown(options=["viridis","magma","cividis"])))

### Predict, then change one control
Does a brighter image necessarily mean a stronger physiological signal? Change display bounds without changing the source.
The palette selector changes the **human-facing numeric display**, while the scalar bounds change model pixels.
These controls preview alternative renderings without overwriting saved reference inputs. Neither creates a new EEG recording. Narrow bounds can clip values; resizing 29×13 into 64×64 creates no new resolution.
Record one example where a rendering change makes the same waveform look substantially different.

In [ ]:
import io, numpy as np
from PIL import Image
pixel = data["pixels"][0]
encoded, decoded = [], []
for level in [0, 9]:
    buffer = io.BytesIO(); Image.fromarray(pixel).save(buffer, format="PNG", compress_level=level)
    encoded.append(buffer.getvalue())
    decoded.append(np.array(Image.open(io.BytesIO(encoded[-1]))))
assert encoded[0] != encoded[1] and np.array_equal(decoded[0], decoded[1])
print("Different PNG bytes, identical decoded pixels:", [len(x) for x in encoded])
studio.write_json(OUTPUT_DIR / "encoding_check.json", {"file_bytes_differ": encoded[0] != encoded[1],
    "pixels_equal": bool(np.array_equal(decoded[0], decoded[1]))})

## Explain and save
1. Trace one image to its EDF hash, window start, numeric power matrix, renderer contract, and pixel hash.
2. Explain why this one person's images cannot establish generalization to new people.
3. State which information is lost through phase removal, channel-power averaging, clipping, and quantization.
4. Save your figures, `contract.json`, manifests, environment, helper bytes, notebook, and reflection before the runtime ends.
**Next:** Notebook 2 fits a small classifier with separate training, validation, and test participants.